In [13]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.stats.api import het_breuschpagan, het_white # тесты на гетероскедастичность
from statsmodels.iolib.summary2 import summary_col, summary_params # вывод результатов тестирования

from scipy.stats import chi2 # chi2-распределение и критические значения
from scipy.stats import t # t-распределение и критические значения
from scipy.stats import f # f-распределение и критические значения
# настройки визуализации
import matplotlib.pyplot as plt

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# 1

In [2]:
# загрузим данные
df = pd.read_csv('sleep75.csv')

In [3]:
# спецификация модели
mod = ols(formula='sleep~1+totwrk+age+I(age**2)+male+smsa+south', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res = mod.fit()

In [4]:
lm, lm_pvalue, fvalue, f_pvalue = het_breuschpagan(resid=res.resid, exog_het=mod.exog)
# LM-статистика и её P-значение 
lm, lm_pvalue

(np.float64(8.310032306323293), np.float64(0.2162580112484363))

In [6]:
mod.df_model

6.0

In [7]:
# Задаём уровень значимости
sign_level = 0.05
# Критическое значение распределения chi2
chi2.ppf(q=1-sign_level, df=mod.df_model)

np.float64(12.591587243743977)

### LM < крит зн => гомоскедастичность

# 2

In [8]:
# загрузим данные
df = pd.read_csv('Labour.csv')

In [17]:
# спецификация модели
mod = ols(formula='np.log(output)~1+np.log(capital)+np.log(labour)+I(np.log(capital)**2)+I(np.log(labour)**2)', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res = mod.fit()

In [18]:
summary_params(res, alpha=0.05).round(3)

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
Intercept,-1.304,0.189,-6.914,0.000,-1.674,-0.934
np.log(capital),0.183,0.017,11.055,0.000,0.151,0.216
np.log(labour),0.515,0.083,6.181,0.000,0.352,0.679
I(np.log(capital) ** 2),0.023,0.005,4.518,0.000,0.013,0.033
I(np.log(labour) ** 2),0.020,0.010,2.112,0.035,0.001,0.039


In [19]:
res_hc = mod.fit(cov_type='HC3')
summary_params(res_hc, alpha=0.05).round(3)

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
Intercept,-1.304,0.493,-2.643,0.008,-2.271,-0.337
np.log(capital),0.183,0.029,6.215,0.000,0.125,0.241
np.log(labour),0.515,0.206,2.497,0.013,0.111,0.920
I(np.log(capital) ** 2),0.023,0.008,2.737,0.006,0.006,0.039
I(np.log(labour) ** 2),0.020,0.021,0.965,0.334,-0.021,0.061


In [20]:
# уровень значимости
sign_level = 0.05
# критическое значение t-распределения
t.ppf(q=1-sign_level/2, df=mod.df_resid)

np.float64(1.9641790265687167)

Значимы все кроме последнего т.к. у последнего |t| > крит знач